## 🎯 Learning Objectives
* Understand the core principles and operational loop of ReAct agents.
* Learn how to implement a basic ReAct agent in LangChain using tools.
* Grasp the concept and utility of structured output agents for reliable data extraction and API interaction.
* Implement a structured output agent in LangChain to enforce a specific JSON schema.
* Analyze the trade-offs, advantages, and typical use cases for both ReAct and structured output agent types.


## Agent Types: ReAct and Structured Output Agents

In the realm of Agentic AI, an "agent" is an intelligent entity that can perceive its environment, make decisions, and take actions to achieve a goal. LangChain provides powerful abstractions to build such agents, and understanding different agent types is crucial for selecting the right approach for your application.

Today, we'll dive into two fundamental and widely used agent types: **ReAct Agents** and **Structured Output Agents**.

### 1. ReAct Agents: Reasoning and Acting

Imagine a seasoned detective solving a complex case. They don't just guess the culprit; they follow a systematic process:

1.  **Observe**: Gather initial clues and information.
2.  **Think (Reason)**: Formulate hypotheses, consider what information is missing, and plan the next step.
3.  **Act**: Interview a witness, search a database, or analyze a piece of evidence.
4.  **Observe (again)**: See the result of their action.
5.  **Think (again)**: Refine hypotheses, decide on the next action based on new observations.

This iterative loop of **Reasoning** and **Acting** is precisely what the **ReAct** (Reasoning and Acting) framework embodies. Developed by Google, ReAct agents leverage a Large Language Model (LLM) to both *reason* about a task (generating `Thought`s) and *act* by using external `Tool`s. The LLM's `Thought` process guides its `Action`, and the `Observation` from the tool's execution feeds back into the LLM, allowing it to refine its reasoning and take subsequent actions until the goal is achieved.

**How it works:**

*   **Thought**: The LLM explains its current reasoning, what it's trying to achieve, and what it plans to do next.
*   **Action**: The LLM decides which tool to use and what input to provide to that tool.
*   **Observation**: The result returned by the tool after its execution.

This loop continues until the LLM determines it has enough information to provide a final `Answer`.

### 2. Structured Output Agents: Precision and Predictability

While ReAct agents excel at open-ended problem-solving, there are many scenarios where you need an agent to produce output in a very specific, machine-readable format – for example, a JSON object conforming to a particular schema, an XML document, or a Pydantic model. This is where **Structured Output Agents** shine.

Think of a meticulous librarian who, after finding a book, doesn't just tell you its title but provides a complete catalog entry: title, author, ISBN, publication date, and genre, all neatly organized into a predefined form. Structured output agents ensure that the LLM's final response adheres to a predefined structure, making it incredibly reliable for downstream processing, API calls, or database interactions.

**How it works:**

*   You define a schema (e.g., using a Pydantic model in Python) that specifies the exact structure and data types of the desired output.
*   The agent is configured to use this schema, often by instructing the LLM through system prompts or function calling mechanisms to generate output that matches the schema.
*   The agent then parses the LLM's response, validating it against the schema and often retrying or correcting if the initial output doesn't conform.

This guarantees that your application receives data in a predictable and usable format, reducing parsing errors and improving the robustness of your LLM-powered workflows.


In [ ]:
import os
from typing import List, Dict, Any
from pydantic import BaseModel, Field

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent, create_structured_output_runnable

# --- Configuration --- #
# Ensure you have your OpenAI API key set as an environment variable
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# For demonstration, we'll use a placeholder if not set, but it won't run without a valid key.
if not os.getenv("OPENAI_API_KEY"):
    print("WARNING: OPENAI_API_KEY environment variable not set. Code will not run without it.")
    # You can uncomment the line below and replace with your key for local testing
    # os.environ["OPENAI_API_KEY"] = "sk-..."

# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("--- Demonstrating ReAct Agent ---")

# --- 1. Define Tools for the ReAct Agent ---

@tool
def calculator(expression: str) -> str:
    """Useful for performing mathematical calculations. Input should be a string mathematical expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error calculating: {e}"

@tool
def word_length_counter(word: str) -> int:
    """Counts the number of characters in a given word."""
    return len(word)

# List of tools available to the agent
tools = [calculator, word_length_counter]

# --- 2. Define the Prompt for the ReAct Agent ---
# This prompt guides the LLM on how to reason and act.
react_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that can use tools to answer questions."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}") # This is where the agent's thoughts and actions are stored
])

# --- 3. Create the ReAct Agent ---
# create_react_agent is a factory function that sets up the ReAct logic.
react_agent = create_react_agent(llm, tools, react_prompt)

# --- 4. Create an AgentExecutor ---
# The AgentExecutor is responsible for running the agent, managing the loop,
# and handling tool execution.
react_agent_executor = AgentExecutor(
    agent=react_agent,
    tools=tools,
    verbose=True, # Set to True to see the thought/action/observation trace
    handle_parsing_errors=True # Helps in robust agent execution
)

# --- 5. Invoke the ReAct Agent ---
print("\nRunning ReAct agent with a calculation task...")
react_result_1 = react_agent_executor.invoke({"input": "What is 123 plus 456 multiplied by 2?"})
print(f"Final ReAct Agent Answer: {react_result_1['output']}")

print("\nRunning ReAct agent with a word length task...")
react_result_2 = react_agent_executor.invoke({"input": "How many letters are in the word 'supercalifragilisticexpialidocious'?"})
print(f"Final ReAct Agent Answer: {react_result_2['output']}")

print("\n--- Demonstrating Structured Output Agent ---")

# --- 1. Define the Pydantic Schema for Structured Output ---
# This schema dictates the exact structure and types of the desired JSON output.
class ProductInfo(BaseModel):
    product_name: str = Field(description="The name of the product.")
    category: str = Field(description="The category the product belongs to (e.g., Electronics, Books, Apparel).")
    price_usd: float = Field(description="The price of the product in USD.")
    features: List[str] = Field(description="A list of key features of the product.")
    available_in_stock: bool = Field(description="True if the product is currently in stock, False otherwise.")

# --- 2. Create the Structured Output Runnable ---
# This utility function creates a runnable that forces the LLM to output JSON
# conforming to the provided Pydantic schema.
structured_output_runnable = create_structured_output_runnable(ProductInfo, llm)

# --- 3. Define a Prompt for the Structured Output Agent ---
# The prompt guides the LLM to extract information relevant to the schema.
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert data extractor. Extract the requested product information into the specified JSON format."),
    ("human", "Extract product information from the following text: {text}")
])

# --- 4. Combine Prompt and Structured Output Runnable ---
# We create a chain to first format the input and then pass it to the structured output runnable.
structured_agent_chain = structured_prompt | structured_output_runnable

# --- 5. Invoke the Structured Output Agent ---
product_description = (
    "Introducing the 'Quantum Leap Smartwatch Pro', a cutting-edge wearable in the Electronics category. "
    "Priced at $299.99, it boasts features like a heart rate monitor, GPS tracking, and 5-day battery life. "
    "Currently, it's out of stock due to high demand, but more units are expected next week."
)

print("\nRunning Structured Output agent...")
structured_result = structured_agent_chain.invoke({"text": product_description})

print("\nStructured Output Agent Result (Pydantic Model):")
print(structured_result)
print(f"Type of result: {type(structured_result)}")
print(f"Product Name: {structured_result.product_name}")
print(f"Category: {structured_result.category}")
print(f"Price: {structured_result.price_usd}")
print(f"Features: {', '.join(structured_result.features)}")
print(f"In Stock: {structured_result.available_in_stock}")

# Example with a different product
product_description_2 = (
    "The 'Galactic Explorer Telescope' is a premium Optics product, perfect for astronomy enthusiasts. "
    "It costs $1250.00 and includes features like 100x magnification, a smartphone adapter, and a sturdy tripod. "
    "It's readily available in stock."
)

print("\nRunning Structured Output agent with a second product...")
structured_result_2 = structured_agent_chain.invoke({"text": product_description_2})

print("\nStructured Output Agent Result (Pydantic Model):")
print(structured_result_2)
print(f"Type of result: {type(structured_result_2)}")
print(f"Product Name: {structured_result_2.product_name}")
print(f"Category: {structured_result_2.category}")
print(f"Price: {structured_result_2.price_usd}")
print(f"Features: {', '.join(structured_result_2.features)}")
print(f"In Stock: {structured_result_2.available_in_stock}")


### Interpreting the Output and Use Cases

#### ReAct Agent Output Interpretation

When you run the ReAct agent with `verbose=True`, you'll observe a detailed trace of its execution:

*   **Thought**: The LLM articulates its reasoning process. It identifies the need for a tool and plans its next action.
*   **Action**: The LLM specifies which tool it intends to use (e.g., `calculator`, `word_length_counter`) and the input it will provide to that tool.
*   **Observation**: This is the actual output returned by the tool after it has been executed. This observation then feeds back into the LLM's context for its next `Thought`.
*   **Final Answer**: Once the LLM determines it has sufficient information, it provides the final answer to the user's query.

This `Thought -> Action -> Observation` loop is the hallmark of ReAct. It provides transparency into the agent's decision-making and allows for complex, multi-step problem-solving.

**Advantages of ReAct Agents:**

*   **Transparency**: The `Thought` process makes the agent's reasoning visible and debuggable.
*   **Problem-Solving**: Excellent for tasks requiring multiple steps, tool usage, and dynamic decision-making.
*   **Flexibility**: Can adapt to novel situations by chaining tools in unforeseen ways.

**Disadvantages and Trade-offs:**

*   **Latency**: Each `Thought-Action-Observation` step involves an LLM call, which can increase overall latency and token usage.
*   **Reliability**: The quality of reasoning depends heavily on the LLM's capabilities and the clarity of the prompt. Poor reasoning can lead to incorrect tool usage or infinite loops.
*   **Tool Selection**: The agent's performance is limited by the quality and relevance of the tools provided.

**Typical Use Cases for ReAct Agents:**

*   **Complex Question Answering**: Answering questions that require looking up information, performing calculations, or interacting with multiple data sources.
*   **Automated Workflows**: Orchestrating a series of steps involving various APIs (e.g., fetching data, processing it, then sending it to another service).
*   **Data Analysis**: Using tools to query databases, perform statistical analysis, and summarize findings.
*   **Code Generation/Execution**: Writing and executing code snippets to test hypotheses or solve programming problems.

#### Structured Output Agent Output Interpretation

For the structured output agent, the key takeaway is the **guaranteed format** of the output. Instead of a free-form text response, the agent returns a Python object (in our case, a Pydantic `ProductInfo` model instance) that strictly adheres to the defined schema.

Notice how the `structured_result` is directly an instance of `ProductInfo`, allowing you to access its attributes (e.g., `structured_result.product_name`, `structured_result.price_usd`) with type safety and auto-completion.

**Advantages of Structured Output Agents:**

*   **Reliability**: Ensures the output always conforms to a predefined structure, critical for downstream processing.
*   **Type Safety**: When combined with Pydantic, it provides strong typing, reducing runtime errors.
*   **Integration**: Ideal for integrating LLMs into existing systems that expect specific data formats (e.g., APIs, databases, UI components).
*   **Reduced Parsing Errors**: Eliminates the need for complex regex or heuristic-based parsing of LLM free-form text.

**Disadvantages and Trade-offs:**

*   **LLM Constraint**: The LLM might struggle to consistently produce the exact schema, especially with complex or ambiguous input, potentially requiring more robust prompting or error handling.
*   **Less Flexible**: Not suitable for open-ended creative tasks where a free-form response is desired.
*   **Overhead**: Defining schemas adds an initial development step.

**Typical Use Cases for Structured Output Agents:**

*   **Data Extraction**: Extracting specific entities (names, dates, prices, addresses) from unstructured text into a structured format.
*   **API Call Generation**: Generating JSON payloads for API requests based on natural language instructions.
*   **Configuration Generation**: Creating configuration files (e.g., YAML, JSON) from user prompts.
*   **Database Interactions**: Formulating structured queries or data inserts based on natural language.
*   **Form Filling**: Populating form fields automatically from a block of text.


### Resources

*   **LangChain Agents Documentation**: [https://python.langchain.com/docs/modules/agents/](https://python.langchain.com/docs/modules/agents/)
*   **LangChain ReAct Agent**: [https://python.langchain.com/docs/modules/agents/agent_types/react](https://python.langchain.com/docs/modules/agents/agent_types/react)
*   **LangChain Structured Output**: [https://python.langchain.com/docs/modules/agents/agent_types/structured_output](https://python.langchain.com/docs/modules/agents/agent_types/structured_output)
*   **Pydantic Documentation**: [https://docs.pydantic.dev/latest/](https://docs.pydantic.dev/latest/)
*   **ReAct: Synergizing Reasoning and Acting in Language Models (Research Paper)**: [https://arxiv.org/abs/2210.03629](https://arxiv.org/abs/2210.03629)
